# Building and testing a synthetic multilevel linear model

This notebook develops a small model for testing the `MultilevelModel` interface. It is intentionally simple: every level has a known solution, but evaluating that level still requires building and solving a sparse linear system.

By the end, we will have implemented and checked all four user-model operations:

1. `sample_randomness()`
2. `couple_inputs()`
3. `build_linear_system()`
4. `quantity_of_interest()`

## 1. What problem are we solving?

At every level $\ell$, we solve the random linear system

$$
A_\ell u_\ell(X)=b_\ell(X),
$$

where $X\sim N(0,1)$ is a scalar random variable and $u_\ell(X)$ is the random solution vector at level $\ell$. The limiting random quantity we want to approximate is

$$
Q(X)=X.
$$

Because this is a manufactured test problem, we choose a known finite-level approximation. Define

$$
q_\ell(X) = X+h_\ell\sqrt{10^{-4}+|X|}.
$$

Here, $X$ is the limiting scalar quantity and

$$
e_\ell(X) = h_\ell\sqrt{10^{-4}+|X|}
$$

is the level-dependent approximation error. To ensure the approximation error decreases as the level resolution
increases, we define the level resolution as

$$
h_\ell=\frac{1}{n_\ell},
$$

where $n_\ell$ is the number of unknowns at level $\ell$:

$$
n_\ell=4\,2^\ell.
$$

The number of unknowns doubles at every level, so $h_\ell$ is halved:

$$
n_0=4,\qquad n_1=8,\qquad n_2=16, \quad \ldots
$$

$$
h_0=\frac14,\qquad h_1=\frac18,\qquad h_2=\frac1{16}, \quad \ldots
$$

Consequently, as $\ell\to\infty$

$$
h_\ell \to 0
$$

and

$$
q_\ell(X)\to X
$$

The desired, known solution of the level-$\ell$ system is the uniform vector

$$
u_\ell^*(X)
=
q_\ell(X)\mathbf{1}_{n_\ell},
$$

<!-- where $\mathbf{1}_{n_\ell}$ is a vector of $n_\ell$ ones. -->

<br>
<!-- 
We are also free to choose $A_\ell$, so we choose the sparse, symmetric positive-definite matrix

$$
A_\ell = (\ell+2)I_{n_\ell}
$$

and construct the right-hand side from the known level solution:

$$
b_\ell(X) = A_\ell u_\ell^*(X).
$$

Therefore, solving

$$
A_\ell u_\ell(X)=b_\ell(X)
$$

should recover

$$
u_\ell(X)=u_\ell^*(X).
$$

As the level increases,

$$
Q_\ell(X)\to Q(X)=X,
$$

and the vector solution approaches the corresponding uniform limiting solution:

$$
u_\ell(X)\to X \, \mathbf{1}_{n_\ell}.
$$ -->



In [ ]:
from dataclasses import dataclass

import numpy as np
from scipy import sparse

from mlmc_linear_systems.linear_solver import (
    LinearSystem,
    direct_solve,
)
from mlmc_linear_systems.model import CoupledInputs, MultilevelModel

## 2. How are $A_\ell$ and $b_\ell$ determined?

This is a **manufactured solution** test. We first choose the exact level solution $u_\ell(X)$, choose a simple matrix $A$, and then calculate the right-hand side.

For this example let $A_\ell$ be given by the sparse, symmetric positive-definite matrix

$$
A_\ell = (\ell+2)I_{n_\ell}
$$

<!-- giving the right-hand side from the known level solution:

$$
b_\ell(X) = A_\ell u_\ell^*(X).
$$ -->

Therefore, solving

$$
A_\ell u_\ell(X)=b_\ell(X)
$$

should recover

$$
u_\ell(X)=u_\ell^*(X).
$$

As the level increases,

$$
Q_\ell(X)\to Q(X)=X,
$$

and the vector solution approaches the corresponding uniform limiting solution:

$$
u_\ell(X)\to X \, \mathbf{1}_{n_\ell}.
$$

The solver should therefore recover the known vector $u_\ell(X)$. The diagonal matrix is not intended to represent a physical operator; it gives us a sparse, SPD system whose correct answer is known exactly.